#Reading From CSV File

In [0]:
df = (
    spark.read
    .option("header", "true").option("inferSchema","true")
    .csv("/Volumes/workspace/bronze/source_system/source_crm/cust_info.csv")
)
df.display()

#Write it to Bronze Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.bronze.crm_cust_info")

In [0]:
%sql
SELECT * FROM workspace.bronze.crm_cust_info


In [0]:
df = (
    spark.read
    .option("header", "true").option("inferSchema","true")
    .csv("/Volumes/workspace/bronze/source_system/source_crm/prd_info.csv")
)
df.write.mode("overwrite").saveAsTable("workspace.bronze.crm_prd_info")

In [0]:

# Define ingestion configuration for multiple source files and target tables
INGESTION_CONFIG = [
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/raw_sources/source_crm/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/raw_sources/source_crm/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/raw_sources/source_crm/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/raw_sources/source_erp/CUST_AZ12.csv",
        "table": "erp_cust_az12"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/raw_sources/source_erp/LOC_A101.csv",
        "table": "erp_loc_a101"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/raw_sources/source_erp/PX_CAT_G1V2.csv",
        "table": "erp_px_cat_g1v2"
    }
]

# Loop through each ingestion config and load CSV files into Bronze Delta tables
for item in INGESTION_CONFIG:
    # Print which source and table is being ingested
    print(f"Ingesting {item['source']} → workspace.bronze.{item['table']}")

    # Read CSV file into Spark DataFrame with header and schema inference
    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(item["path"])
    )

    # Write DataFrame to Delta table in overwrite mode
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .saveAsTable(f"workspace.bronze.{item['table']}")
    )